In [1]:
import nltk
nltk.download('book')
from nltk.book import *
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

[nltk_data] Downloading collection 'book'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package brown to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/brown.zip.
[nltk_data]    | Downloading package chat80 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/chat80.zip.
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package conll2000 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/conll2000.zip.
[nltk_data]    | Downloading package conll2002 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/conll2002.zip.
[nltk_data]    | Downloading package dependency_treebank to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping corpora/dependency_treebank.zip.
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


# Collocations (Colocaciones)

* Son secuencias de palabras que suelen ocurrir en textos o conversaciones con una **frecuencia inusualmente alta** [NLTK doc](http://www.nltk.org/book/ch01.html)
* Las colocaciones de una palabra son declaraciones formales de donde suele ubicarse tipicamente esa palabra [Manning & Schütze, 1990, Foundations of Statistical Natural Language Processing, Capítulo 6](https://nlp.stanford.edu/fsnlp/)

In [2]:
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
stop_words = set(stopwords.words('english'))
words = [word.lower() for word in text1 if word.isalpha()]
filtered_words = [word for word in words if word not in stop_words]
print(filtered_words[:10])

['moby', 'dick', 'herman', 'melville', 'etymology', 'supplied', 'late', 'consumptive', 'usher', 'grammar']


In [5]:
filtered_bigrams = list(bigrams(filtered_words))
filtered_bigrams[:10]

[('moby', 'dick'),
 ('dick', 'herman'),
 ('herman', 'melville'),
 ('melville', 'etymology'),
 ('etymology', 'supplied'),
 ('supplied', 'late'),
 ('late', 'consumptive'),
 ('consumptive', 'usher'),
 ('usher', 'grammar'),
 ('grammar', 'school')]

In [7]:
filtered_bigram_dist = FreqDist(filtered_bigrams)
filtered_bigram_dist

FreqDist({('sperm', 'whale'): 182, ('white', 'whale'): 106, ('moby', 'dick'): 84, ('old', 'man'): 81, ('captain', 'ahab'): 64, ('right', 'whale'): 57, ('mast', 'head'): 49, ('whale', 'ship'): 37, ('mast', 'heads'): 37, ('ye', 'see'): 37, ...})

# CREAMOS UN DATAFRAME CON LOS VALORES DE FILTRED_BIGRAM_DIST

In [9]:
threshold = 2
filtered_words = [word for word in filtered_words if len(word)>threshold]
filtered_word_dist = FreqDist(filtered_words)
filtered_word_dist

FreqDist({'whale': 1226, 'one': 921, 'like': 647, 'upon': 566, 'man': 527, 'ship': 518, 'ahab': 511, 'sea': 455, 'old': 450, 'would': 432, ...})

In [10]:
df = pd.DataFrame()
df['bi_gram'] = list(set(filtered_bigrams))
df['word_0'] = df['bi_gram'].apply(lambda x: x[0])
df['word_1'] = df['bi_gram'].apply(lambda x: x[1])
df['bi_gram_freq'] = df['bi_gram'].apply(lambda x: filtered_bigram_dist[x])
df['word_0_freq'] = df['word_0'].apply(lambda x: filtered_word_dist[x])
df['word_1_freq'] = df['word_1'].apply(lambda x: filtered_word_dist[x])
df

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq
0,"(though, whole)",though,whole,1,384,137
1,"(blast, think)",blast,think,1,15,122
2,"(lifted, final)",lifted,final,1,17,29
3,"(square, timber)",square,timber,1,14,6
4,"(nervous, step)",nervous,step,1,9,22
...,...,...,...,...,...,...
98017,"(work, lonely)",work,lonely,1,65,12
98018,"(prefers, drink)",prefers,drink,1,2,18
98019,"(tail, forty)",tail,forty,1,80,33
98020,"(concerns, specksynder)",concerns,specksynder,1,1,3


# APLICAMOS FORMULA PARA CALCULAR COLOCACIONES

# Pointwise Mutual Information (PMI)
Una métrica basada en _teoria de la información_ para encontrar **Collocations**.

$$
PMI = \log\left(\frac{P(w_1, w_2)}{P(w_1)P(w_2)}\right)
$$

In [11]:
df['PMI'] = df[['bi_gram_freq', 'word_0_freq', 'word_1_freq']].apply(lambda x:np.log2(x.values[0]/(x.values[1]*x.values[2])), axis = 1)
df['log(bi_gram_freq)'] = df['bi_gram_freq'].apply(lambda x: np.log2(x))
df

<ipython-input-11-9206388c5b52>:1: RuntimeWarning: divide by zero encountered in scalar divide
  df['PMI'] = df[['bi_gram_freq', 'word_0_freq', 'word_1_freq']].apply(lambda x:np.log2(x.values[0]/(x.values[1]*x.values[2])), axis = 1)


,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI,log(bi_gram_freq)
0,"(though, whole)",though,whole,1,384,137,-15.682995,0.0
1,"(blast, think)",blast,think,1,15,122,-10.837628,0.0
2,"(lifted, final)",lifted,final,1,17,29,-8.945444,0.0
3,"(square, timber)",square,timber,1,14,6,-6.392317,0.0
4,"(nervous, step)",nervous,step,1,9,22,-7.629357,0.0
...,...,...,...,...,...,...,...,...
98017,"(work, lonely)",work,lonely,1,65,12,-9.607330,0.0
98018,"(prefers, drink)",prefers,drink,1,2,18,-5.169925,0.0
98019,"(tail, forty)",tail,forty,1,80,33,-11.366322,0.0
98020,"(concerns, specksynder)",concerns,specksynder,1,1,3,-1.584963,0.0


In [12]:
df.sort_values(by = 'PMI', ascending=False)

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI,log(bi_gram_freq)
91123,"(ho, cried)",ho,cried,2,0,155,inf,1.0
91044,"(assured, us)",assured,us,1,6,0,inf,0.0
91043,"(grew, us)",grew,us,1,14,0,inf,0.0
41471,"(hope, ye)",hope,ye,1,34,0,inf,0.0
41485,"(poles, ye)",poles,ye,1,12,0,inf,0.0
...,...,...,...,...,...,...,...,...
85288,"(like, like)",like,like,1,647,647,-18.675244,0.0
31783,"(though, whale)",though,whale,1,384,1226,-18.844706,0.0
10566,"(man, one)",man,one,1,527,921,-18.888716,0.0
11717,"(would, whale)",would,whale,1,432,1226,-19.014631,0.0


# CALCULO DE PMI PARA COLOCACIONES USANDO NTLK

In [14]:
from  nltk.collocations import *
bigram_measures = nltk.collocations.BigramAssocMeasures()
finder = BigramCollocationFinder.from_words(filtered_words)
finder.apply_freq_filter(20)


In [15]:
finder.nbest(bigram_measures.pmi,10)

[('moby', 'dick'),
 ('quarter', 'deck'),
 ('mast', 'heads'),
 ('thou', 'art'),
 ('aye', 'aye'),
 ('never', 'mind'),
 ('captain', 'peleg'),
 ('mast', 'head'),
 ('sperm', 'whale'),
 ('cried', 'stubb')]